# Notes Workflow Test Suite

This notebook tests the complete note workflow:

```
DRAFT ←──────────────────────┐
   │                         │ (Rejected)
   ├──→ IN_REVIEW ───────────┤
   │         │               │
   │         ↓ (Approved)    │
   ├──→ READY ───→ RELEASED ─┴──→ ARCHIVED
   │    (Teacher shortcut)
```

## Test Scenarios
1. **Teacher Workflow**: DRAFT → READY → RELEASED (skip review)
2. **Content Creator Workflow**: DRAFT → IN_REVIEW → READY → RELEASED
3. **Rejection Workflow**: DRAFT → IN_REVIEW → DRAFT (with feedback)
4. **Archive Workflow**: Any state → ARCHIVED
5. **Student Visibility**: Can only see RELEASED notes

---

## Setup

In [1]:
import requests
import json
import uuid
from datetime import datetime

# Service URLs
AUTH_URL = "http://localhost:8081"
TENANT_URL = "http://localhost:8082"
NOTES_URL = "http://localhost:8088"

# Test data storage
test_data = {
    "tenant_id": None,
    "teacher": {},
    "content_creator": {},
    "admin": {},
    "student": {},
    "notes": []
}

def print_response(response, label="Response"):
    """Pretty print API response"""
    print(f"{label} - Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except:
        print(response.text)

print("Setup complete!")

Setup complete!


### Create Test Tenant

In [2]:
# Get or create test tenant
tenant_key = "workflow-test-school"

print("Checking for existing tenant...")
response = requests.get(f"{TENANT_URL}/v1/tenants")

existing_tenant = None
if response.status_code == 200:
    tenants = response.json()
    for tenant in tenants:
        if tenant.get("tenantKey") == tenant_key:
            existing_tenant = tenant
            break

if existing_tenant:
    test_data["tenant_id"] = existing_tenant["id"]
    print(f"Using existing tenant: {existing_tenant['name']} (ID: {existing_tenant['id']})")
else:
    response = requests.post(
        f"{TENANT_URL}/v1/tenants",
        json={
            "tenantKey": tenant_key,
            "name": "Workflow Test School",
            "status": "ACTIVE"
        }
    )
    if response.status_code in [200, 201]:
        tenant = response.json()
        test_data["tenant_id"] = tenant["id"]
        print(f"Created tenant: {tenant['name']} (ID: {tenant['id']})")
    else:
        print(f"Failed to create tenant: {response.text}")

print(f"\nTenant ID: {test_data['tenant_id']}")

Checking for existing tenant...
Created tenant: Workflow Test School (ID: b92e9e4d-ef18-4802-9fb3-ddcde97279a7)

Tenant ID: b92e9e4d-ef18-4802-9fb3-ddcde97279a7


### Create Test Users

In [3]:
# Create test users with different roles
tenant_id = test_data["tenant_id"]
unique_suffix = uuid.uuid4().hex[:6]

users_to_create = [
    ("teacher", f"teacher_{unique_suffix}@test.edu", "Test Teacher"),
    ("content_creator", f"creator_{unique_suffix}@test.edu", "Content Creator"),
    ("admin", f"admin_{unique_suffix}@test.edu", "School Admin"),
    ("student", f"student_{unique_suffix}@test.edu", "Test Student")
]

print("Creating test users...")
print("-" * 50)

for role, email, name in users_to_create:
    response = requests.post(
        f"{AUTH_URL}/auth/signup",
        json={
            "email": email,
            "password": "Test@123",
            "tenantId": str(tenant_id),
            "displayName": name
        }
    )
    
    if response.status_code in [200, 201]:
        data = response.json()
        test_data[role] = {
            "userId": data.get("userId"),
            "accessToken": data.get("accessToken"),
            "email": email,
            "name": name
        }
        print(f"✓ Created {role}: {name} (ID: {data.get('userId')})")
    else:
        print(f"✗ Failed to create {role}: {response.text}")

print("-" * 50)
print("Users created!")

Creating test users...
--------------------------------------------------
✓ Created teacher: Test Teacher (ID: 865a1cf4-501e-4f51-9e23-f43be8bf9ec4)
✓ Created content_creator: Content Creator (ID: c09a23cf-063e-4ca9-b054-0be16de70b4e)
✓ Created admin: School Admin (ID: bf9bafea-9502-4f69-b514-3bf87439987c)
✓ Created student: Test Student (ID: f7dd4651-003f-40e3-94b2-b53dbae5b9dc)
--------------------------------------------------
Users created!


---

## Test 1: Teacher Workflow (DRAFT → READY → RELEASED)

Teachers can skip the review process and directly mark notes as READY.

### 1.1 Teacher Creates a DRAFT Note

In [4]:
teacher = test_data["teacher"]
tenant_id = test_data["tenant_id"]

print("Test 1.1: Teacher creates a DRAFT note")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {teacher['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={
        "title": "Math: Introduction to Algebra",
        "summary": "Basic algebraic concepts for beginners",
        "contentMd": "# Introduction to Algebra\n\nAlgebra is a branch of mathematics...",
        "createdBy": str(teacher['userId']),
        "tags": ["math", "algebra", "grade-8"]
    }
)

if response.status_code in [200, 201]:
    note = response.json()
    teacher_note_id = note.get("id")
    test_data["notes"].append({"id": teacher_note_id, "tenant_id": tenant_id})
    
    print(f"Note ID: {teacher_note_id}")
    print(f"Status: {note.get('status')}")
    print(f"Title: {note.get('title')}")
    
    if note.get('status') == 'DRAFT':
        print("\n✓ [PASS] Note created with DRAFT status")
    else:
        print(f"\n✗ [FAIL] Expected DRAFT, got {note.get('status')}")
else:
    print(f"Failed: {response.text}")
    teacher_note_id = None

Test 1.1: Teacher creates a DRAFT note
Note ID: 7ad5d87c-a874-47a1-912c-e533ee6061fa
Status: DRAFT
Title: Math: Introduction to Algebra

✓ [PASS] Note created with DRAFT status


### 1.2 Student Cannot See DRAFT Note

In [5]:
student = test_data["student"]

print("Test 1.2: Student tries to see notes (should NOT see DRAFT)")
print("=" * 50)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {student['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    }
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", [])
    
    draft_visible = any(n.get("id") == teacher_note_id for n in notes)
    
    print(f"Student sees {len(notes)} note(s)")
    
    if not draft_visible:
        print("\n✓ [PASS] Student cannot see DRAFT note")
    else:
        print("\n✗ [FAIL] Student can see DRAFT note!")
else:
    print(f"Error: {response.text}")

Test 1.2: Student tries to see notes (should NOT see DRAFT)
Student sees 0 note(s)

✓ [PASS] Student cannot see DRAFT note


### 1.3 Teacher Marks Note as READY (Skip Review)

In [6]:
print("Test 1.3: Teacher marks note as READY (skipping review)")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes/{teacher_note_id}/mark-ready",
    headers={
        "Authorization": f"Bearer {teacher['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    params={"readyBy": str(teacher['userId'])}
)

if response.status_code == 200:
    note = response.json()
    print(f"Note ID: {note.get('id')}")
    print(f"Status: {note.get('status')}")
    print(f"Reviewed By: {note.get('reviewedBy')}")
    print(f"Reviewed At: {note.get('reviewedAt')}")
    
    if note.get('status') == 'READY':
        print("\n✓ [PASS] Note marked as READY")
    else:
        print(f"\n✗ [FAIL] Expected READY, got {note.get('status')}")
else:
    print(f"Failed: {response.text}")

Test 1.3: Teacher marks note as READY (skipping review)
Note ID: 7ad5d87c-a874-47a1-912c-e533ee6061fa
Status: READY
Reviewed By: 865a1cf4-501e-4f51-9e23-f43be8bf9ec4
Reviewed At: 2026-01-26T08:40:08.359979Z

✓ [PASS] Note marked as READY


### 1.4 Student Still Cannot See READY Note

In [7]:
print("Test 1.4: Student tries to see notes (should NOT see READY)")
print("=" * 50)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {student['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    }
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", [])
    
    ready_visible = any(n.get("id") == teacher_note_id for n in notes)
    
    print(f"Student sees {len(notes)} note(s)")
    
    if not ready_visible:
        print("\n✓ [PASS] Student cannot see READY note (not yet released)")
    else:
        print("\n✗ [FAIL] Student can see READY note before release!")
else:
    print(f"Error: {response.text}")

Test 1.4: Student tries to see notes (should NOT see READY)
Student sees 0 note(s)

✓ [PASS] Student cannot see READY note (not yet released)


### 1.5 Teacher Releases Note to Students

In [8]:
print("Test 1.5: Teacher releases note to students")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes/{teacher_note_id}/release",
    headers={
        "Authorization": f"Bearer {teacher['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={"releasedBy": str(teacher['userId'])}
)

if response.status_code == 200:
    note = response.json()
    print(f"Note ID: {note.get('id')}")
    print(f"Status: {note.get('status')}")
    print(f"Released By: {note.get('releasedBy')}")
    print(f"Released At: {note.get('releasedAt')}")
    
    if note.get('status') == 'RELEASED':
        print("\n✓ [PASS] Note released successfully")
    else:
        print(f"\n✗ [FAIL] Expected RELEASED, got {note.get('status')}")
else:
    print(f"Failed: {response.text}")

Test 1.5: Teacher releases note to students
Note ID: 7ad5d87c-a874-47a1-912c-e533ee6061fa
Status: RELEASED
Released By: 865a1cf4-501e-4f51-9e23-f43be8bf9ec4
Released At: 2026-01-26T08:40:25.015788Z

✓ [PASS] Note released successfully


### 1.6 Student CAN Now See RELEASED Note

In [9]:
print("Test 1.6: Student can now see RELEASED note")
print("=" * 50)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {student['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    }
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", [])
    
    released_visible = any(n.get("id") == teacher_note_id for n in notes)
    
    print(f"Student sees {len(notes)} note(s):")
    for n in notes:
        print(f"  - {n.get('title')} (Status: {n.get('status')})")
    
    if released_visible:
        print("\n✓ [PASS] Student can now see RELEASED note!")
    else:
        print("\n✗ [FAIL] Student cannot see RELEASED note")
else:
    print(f"Error: {response.text}")

Test 1.6: Student can now see RELEASED note
Student sees 1 note(s):
  - Math: Introduction to Algebra (Status: RELEASED)

✓ [PASS] Student can now see RELEASED note!


---

## Test 2: Content Creator Workflow (DRAFT → IN_REVIEW → READY → RELEASED)

Content creators must go through the review process.

### 2.1 Content Creator Creates a DRAFT Note

In [10]:
creator = test_data["content_creator"]

print("Test 2.1: Content Creator creates a DRAFT note")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {creator['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={
        "title": "Science: The Solar System",
        "summary": "An overview of planets in our solar system",
        "contentMd": "# The Solar System\n\nOur solar system consists of...",
        "createdBy": str(creator['userId']),
        "tags": ["science", "astronomy", "grade-6"]
    }
)

if response.status_code in [200, 201]:
    note = response.json()
    creator_note_id = note.get("id")
    test_data["notes"].append({"id": creator_note_id, "tenant_id": tenant_id})
    
    print(f"Note ID: {creator_note_id}")
    print(f"Status: {note.get('status')}")
    
    if note.get('status') == 'DRAFT':
        print("\n✓ [PASS] Note created with DRAFT status")
    else:
        print(f"\n✗ [FAIL] Expected DRAFT, got {note.get('status')}")
else:
    print(f"Failed: {response.text}")
    creator_note_id = None

Test 2.1: Content Creator creates a DRAFT note
Note ID: 96a078be-82fa-46e7-af2c-aec44dbfda2f
Status: DRAFT

✓ [PASS] Note created with DRAFT status


### 2.2 Content Creator Submits Note for Review

In [11]:
print("Test 2.2: Content Creator submits note for review")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes/{creator_note_id}/submit-for-review",
    headers={
        "Authorization": f"Bearer {creator['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={
        "submittedBy": str(creator['userId']),
        "submissionNotes": "Please review this content for grade 6 students"
    }
)

if response.status_code == 200:
    note = response.json()
    print(f"Note ID: {note.get('id')}")
    print(f"Status: {note.get('status')}")
    print(f"Submitted By: {note.get('submittedBy')}")
    print(f"Submitted At: {note.get('submittedAt')}")
    
    if note.get('status') == 'IN_REVIEW':
        print("\n✓ [PASS] Note submitted for review (IN_REVIEW)")
    else:
        print(f"\n✗ [FAIL] Expected IN_REVIEW, got {note.get('status')}")
else:
    print(f"Failed: {response.text}")

Test 2.2: Content Creator submits note for review
Note ID: 96a078be-82fa-46e7-af2c-aec44dbfda2f
Status: IN_REVIEW
Submitted By: c09a23cf-063e-4ca9-b054-0be16de70b4e
Submitted At: 2026-01-26T08:40:52.110597Z

✓ [PASS] Note submitted for review (IN_REVIEW)


### 2.3 Admin Approves the Note

In [12]:
admin = test_data["admin"]

print("Test 2.3: Admin approves the note")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes/{creator_note_id}/approve",
    headers={
        "Authorization": f"Bearer {admin['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={
        "approvedBy": str(admin['userId']),
        "approvalNotes": "Content looks good. Approved for release."
    }
)

if response.status_code == 200:
    note = response.json()
    print(f"Note ID: {note.get('id')}")
    print(f"Status: {note.get('status')}")
    print(f"Reviewed By: {note.get('reviewedBy')}")
    print(f"Reviewed At: {note.get('reviewedAt')}")
    
    if note.get('status') == 'READY':
        print("\n✓ [PASS] Note approved and marked READY")
    else:
        print(f"\n✗ [FAIL] Expected READY, got {note.get('status')}")
else:
    print(f"Failed: {response.text}")

Test 2.3: Admin approves the note
Note ID: 96a078be-82fa-46e7-af2c-aec44dbfda2f
Status: READY
Reviewed By: bf9bafea-9502-4f69-b514-3bf87439987c
Reviewed At: 2026-01-26T08:40:56.686636Z

✓ [PASS] Note approved and marked READY


### 2.4 Teacher Releases the Approved Note

In [13]:
print("Test 2.4: Teacher releases the approved note")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes/{creator_note_id}/release",
    headers={
        "Authorization": f"Bearer {teacher['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={"releasedBy": str(teacher['userId'])}
)

if response.status_code == 200:
    note = response.json()
    print(f"Status: {note.get('status')}")
    print(f"Released At: {note.get('releasedAt')}")
    
    if note.get('status') == 'RELEASED':
        print("\n✓ [PASS] Note released to students")
    else:
        print(f"\n✗ [FAIL] Expected RELEASED, got {note.get('status')}")
else:
    print(f"Failed: {response.text}")

Test 2.4: Teacher releases the approved note
Status: RELEASED
Released At: 2026-01-26T08:40:59.901935Z

✓ [PASS] Note released to students


---

## Test 3: Rejection Workflow (DRAFT → IN_REVIEW → DRAFT)

When admin rejects a note, it goes back to DRAFT with feedback.

### 3.1 Create and Submit Note for Review

In [14]:
print("Test 3.1: Create and submit a note for review")
print("=" * 50)

# Create note
response = requests.post(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {creator['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={
        "title": "History: World War II (Needs Revision)",
        "summary": "Overview of WWII - has some errors",
        "contentMd": "# World War II\n\nSome incorrect information here...",
        "createdBy": str(creator['userId'])
    }
)

if response.status_code in [200, 201]:
    note = response.json()
    reject_note_id = note.get("id")
    test_data["notes"].append({"id": reject_note_id, "tenant_id": tenant_id})
    print(f"Created note: {reject_note_id}")
    
    # Submit for review
    response = requests.post(
        f"{NOTES_URL}/notes/{reject_note_id}/submit-for-review",
        headers={
            "Authorization": f"Bearer {creator['accessToken']}",
            "X-Tenant-Id": str(tenant_id)
        },
        json={"submittedBy": str(creator['userId'])}
    )
    
    if response.status_code == 200:
        note = response.json()
        print(f"Status: {note.get('status')}")
        print("\n✓ [PASS] Note submitted for review")
else:
    print(f"Failed: {response.text}")
    reject_note_id = None

Test 3.1: Create and submit a note for review
Created note: 6a57023e-4903-49a1-9b4f-0da861ade1bf
Status: IN_REVIEW

✓ [PASS] Note submitted for review


### 3.2 Admin Rejects the Note with Feedback

In [15]:
print("Test 3.2: Admin rejects the note with feedback")
print("=" * 50)

response = requests.post(
    f"{NOTES_URL}/notes/{reject_note_id}/reject",
    headers={
        "Authorization": f"Bearer {admin['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={
        "rejectedBy": str(admin['userId']),
        "rejectionReason": "Please fix the following issues:\n1. Incorrect date for D-Day\n2. Missing information about Pacific theater\n3. Need more sources cited"
    }
)

if response.status_code == 200:
    note = response.json()
    print(f"Note ID: {note.get('id')}")
    print(f"Status: {note.get('status')}")
    print(f"Reviewed By: {note.get('reviewedBy')}")
    print(f"Rejection Reason: {note.get('rejectionReason')}")
    
    if note.get('status') == 'DRAFT' and note.get('rejectionReason'):
        print("\n✓ [PASS] Note rejected and returned to DRAFT with feedback")
    else:
        print(f"\n✗ [FAIL] Expected DRAFT with rejection reason")
else:
    print(f"Failed: {response.text}")

Test 3.2: Admin rejects the note with feedback
Note ID: 6a57023e-4903-49a1-9b4f-0da861ade1bf
Status: DRAFT
Reviewed By: bf9bafea-9502-4f69-b514-3bf87439987c
Rejection Reason: Please fix the following issues:
1. Incorrect date for D-Day
2. Missing information about Pacific theater
3. Need more sources cited

✓ [PASS] Note rejected and returned to DRAFT with feedback


---

## Test 4: Archive Workflow

Notes can be archived (soft deleted) from any state.

### 4.1 Archive a RELEASED Note

In [16]:
print("Test 4.1: Archive a RELEASED note")
print("=" * 50)

# Create and release a note first
response = requests.post(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {teacher['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    json={
        "title": "Old Note: To Be Archived",
        "summary": "This note will be archived",
        "contentMd": "Content...",
        "createdBy": str(teacher['userId'])
    }
)

if response.status_code in [200, 201]:
    note = response.json()
    archive_note_id = note.get("id")
    test_data["notes"].append({"id": archive_note_id, "tenant_id": tenant_id})
    
    # Mark ready
    requests.post(
        f"{NOTES_URL}/notes/{archive_note_id}/mark-ready",
        headers={"X-Tenant-Id": str(tenant_id)},
        params={"readyBy": str(teacher['userId'])}
    )
    
    # Release
    requests.post(
        f"{NOTES_URL}/notes/{archive_note_id}/release",
        headers={"X-Tenant-Id": str(tenant_id)},
        json={"releasedBy": str(teacher['userId'])}
    )
    
    print(f"Created and released note: {archive_note_id}")
    
    # Now archive it
    response = requests.post(
        f"{NOTES_URL}/notes/{archive_note_id}/archive",
        headers={
            "Authorization": f"Bearer {teacher['accessToken']}",
            "X-Tenant-Id": str(tenant_id)
        },
        json={
            "archivedBy": str(teacher['userId']),
            "archiveReason": "Content is outdated and no longer relevant"
        }
    )
    
    if response.status_code == 200:
        note = response.json()
        print(f"Status: {note.get('status')}")
        print(f"Archived By: {note.get('archivedBy')}")
        print(f"Archive Reason: {note.get('archiveReason')}")
        
        if note.get('status') == 'ARCHIVED':
            print("\n✓ [PASS] Note archived successfully")
        else:
            print(f"\n✗ [FAIL] Expected ARCHIVED, got {note.get('status')}")
    else:
        print(f"Failed to archive: {response.text}")
else:
    print(f"Failed to create note: {response.text}")

Test 4.1: Archive a RELEASED note
Created and released note: 01683c06-d2d1-4508-826b-227a55e63ecb
Status: ARCHIVED
Archived By: 865a1cf4-501e-4f51-9e23-f43be8bf9ec4
Archive Reason: Content is outdated and no longer relevant

✓ [PASS] Note archived successfully


### 4.2 Archived Note Not Visible to Students

In [17]:
print("Test 4.2: Archived note should not be visible to students")
print("=" * 50)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {student['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    }
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", [])
    
    archived_visible = any(n.get("id") == archive_note_id for n in notes)
    
    print(f"Student sees {len(notes)} note(s)")
    
    if not archived_visible:
        print("\n✓ [PASS] Archived note is not visible to students")
    else:
        print("\n✗ [FAIL] Archived note is still visible!")
else:
    print(f"Error: {response.text}")

Test 4.2: Archived note should not be visible to students
Student sees 2 note(s)

✓ [PASS] Archived note is not visible to students


---

## Test 5: Teacher Can See Own Notes in All States

In [18]:
print("Test 5: Teacher can see their own notes (all states)")
print("=" * 50)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers={
        "Authorization": f"Bearer {teacher['accessToken']}",
        "X-Tenant-Id": str(tenant_id)
    },
    params={"createdBy": str(teacher['userId'])}
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", [])
    
    print(f"Teacher sees {len(notes)} of their own note(s):")
    for n in notes:
        print(f"  - {n.get('title')[:40]}... (Status: {n.get('status')})")
    
    if len(notes) > 0:
        print("\n✓ [PASS] Teacher can see their own notes")
    else:
        print("\n✗ [FAIL] Teacher cannot see their notes")
else:
    print(f"Error: {response.text}")

Test 5: Teacher can see their own notes (all states)
Teacher sees 2 of their own note(s):
  - Old Note: To Be Archived... (Status: ARCHIVED)
  - Math: Introduction to Algebra... (Status: RELEASED)

✓ [PASS] Teacher can see their own notes


---

## Test Summary

In [19]:
print("\n" + "=" * 60)
print("TEST SUMMARY")
print("=" * 60)
print("""
Workflow Tests Completed:

1. Teacher Workflow (DRAFT → READY → RELEASED)
   - Teacher creates DRAFT note
   - Student cannot see DRAFT
   - Teacher marks as READY (skip review)
   - Student cannot see READY
   - Teacher releases note
   - Student can see RELEASED

2. Content Creator Workflow (DRAFT → IN_REVIEW → READY → RELEASED)
   - Creator creates DRAFT
   - Creator submits for review
   - Admin approves
   - Teacher releases

3. Rejection Workflow (DRAFT → IN_REVIEW → DRAFT)
   - Note submitted for review
   - Admin rejects with feedback
   - Note returns to DRAFT

4. Archive Workflow
   - RELEASED note archived
   - Archived note not visible to students

5. Teacher Visibility
   - Teacher can see own notes in all states
""")
print("=" * 60)


TEST SUMMARY

Workflow Tests Completed:

1. Teacher Workflow (DRAFT → READY → RELEASED)
   - Teacher creates DRAFT note
   - Student cannot see DRAFT
   - Teacher marks as READY (skip review)
   - Student cannot see READY
   - Teacher releases note
   - Student can see RELEASED

2. Content Creator Workflow (DRAFT → IN_REVIEW → READY → RELEASED)
   - Creator creates DRAFT
   - Creator submits for review
   - Admin approves
   - Teacher releases

3. Rejection Workflow (DRAFT → IN_REVIEW → DRAFT)
   - Note submitted for review
   - Admin rejects with feedback
   - Note returns to DRAFT

4. Archive Workflow
   - RELEASED note archived
   - Archived note not visible to students

5. Teacher Visibility
   - Teacher can see own notes in all states



---

## Cleanup

In [20]:
print("Cleaning up test resources...")
print("-" * 50)

# Delete notes
for note_info in test_data.get("notes", []):
    note_id = note_info.get("id")
    tenant_id = note_info.get("tenant_id")
    if note_id:
        response = requests.delete(
            f"{NOTES_URL}/notes/{note_id}",
            headers={"X-Tenant-Id": str(tenant_id)}
        )
        print(f"Deleted note {note_id} - Status: {response.status_code}")

# Delete users
for role in ["teacher", "content_creator", "admin", "student"]:
    user = test_data.get(role, {})
    user_id = user.get("userId")
    if user_id:
        response = requests.delete(f"{AUTH_URL}/auth/users/{user_id}")
        print(f"Deleted {role} {user_id} - Status: {response.status_code}")

print("-" * 50)
print("Cleanup complete!")

Cleaning up test resources...
--------------------------------------------------
Deleted note 7ad5d87c-a874-47a1-912c-e533ee6061fa - Status: 204
Deleted note 96a078be-82fa-46e7-af2c-aec44dbfda2f - Status: 204
Deleted note 6a57023e-4903-49a1-9b4f-0da861ade1bf - Status: 204
Deleted note 01683c06-d2d1-4508-826b-227a55e63ecb - Status: 204
Deleted teacher 865a1cf4-501e-4f51-9e23-f43be8bf9ec4 - Status: 204
Deleted content_creator c09a23cf-063e-4ca9-b054-0be16de70b4e - Status: 204
Deleted admin bf9bafea-9502-4f69-b514-3bf87439987c - Status: 204
Deleted student f7dd4651-003f-40e3-94b2-b53dbae5b9dc - Status: 204
--------------------------------------------------
Cleanup complete!
